In [3]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from sklearn.metrics import r2_score
from sklearn import datasets
from sklearn.model_selection import train_test_split

In [4]:
df = datasets.load_diabetes(as_frame=True).frame

In [5]:
df.head()

,age,sex,bmi,bp,s1,s2,s3,s4,s5,s6,target
0,0.038076,0.050680,0.061696,0.021872,-0.044223,-0.034821,-0.043401,-0.002592,0.019907,-0.017646,151.0
1,-0.001882,-0.044642,-0.051474,-0.026328,-0.008449,-0.019163,0.074412,-0.039493,-0.068332,-0.092204,75.0
2,0.085299,0.050680,0.044451,-0.005670,-0.045599,-0.034194,-0.032356,-0.002592,0.002861,-0.025930,141.0
3,-0.089063,-0.044642,-0.011595,-0.036656,0.012191,0.024991,-0.036038,0.034309,0.022688,-0.009362,206.0
4,0.005383,-0.044642,-0.036385,0.021872,0.003935,0.015596,0.008142,-0.002592,-0.031988,-0.046641,135.0


In [6]:
X = df.drop("target",axis=1)
y = df["target"]

In [7]:
X.head()

,age,sex,bmi,bp,s1,s2,s3,s4,s5,s6
0,0.038076,0.050680,0.061696,0.021872,-0.044223,-0.034821,-0.043401,-0.002592,0.019907,-0.017646
1,-0.001882,-0.044642,-0.051474,-0.026328,-0.008449,-0.019163,0.074412,-0.039493,-0.068332,-0.092204
2,0.085299,0.050680,0.044451,-0.005670,-0.045599,-0.034194,-0.032356,-0.002592,0.002861,-0.025930
3,-0.089063,-0.044642,-0.011595,-0.036656,0.012191,0.024991,-0.036038,0.034309,0.022688,-0.009362
4,0.005383,-0.044642,-0.036385,0.021872,0.003935,0.015596,0.008142,-0.002592,-0.031988,-0.046641


In [8]:
y.head()

0    151.0
1     75.0
2    141.0
3    206.0
4    135.0
Name: target, dtype: float64

In [10]:

X_train, X_test, y_train, y_test = train_test_split(
    X,y,test_size=0.3, random_state=42)

In [11]:
# scale 'y'
y_scaler = StandardScaler()
y_train_scaled = y_scaler.fit_transform(y_train.values.reshape(-1,1)).ravel()
y_test_scaled = y_scaler.transform(y_test.values.reshape(-1,1)).ravel()

In [12]:
svr = SVR()
svr.fit(X_train, y_train_scaled)

SVR()

In [17]:
y_train_pred_scaled = svr.predict(X_train)
y_test_pred_scaled = svr.predict(X_test)

In [18]:
print("Train r2:", r2_score(y_train_scaled, y_train_pred_scaled))
print("Test r2:", r2_score(y_test_scaled, y_test_pred_scaled))

Train r2: 0.6596361676267712
Test r2: 0.48844443151651906


# Hyper Parameter Tuning

In [19]:
from sklearn.model_selection import GridSearchCV

In [24]:
# using GridSearchCV

svr = SVR()

params_grid = {
    "kernel":["rbf","linear"],
    "C":[0.5, 1, 2, 3, 4, 5, 10, 50, 100],
    "epsilon":[0.01, 0.1, 0.2, 0.3, 0.5]
}

grid = GridSearchCV(
    svr,
    params_grid,
    scoring="r2",
    cv=5
)

grid.fit(X_train, y_train_scaled)

print("best params:", grid.best_params_)

best params: {'C': 10, 'epsilon': 0.1, 'kernel': 'linear'}


In [26]:
best_model = SVR(C=10, epsilon=0.1, kernel='linear')
best_model.fit(X_train, y_train_scaled)

y_train_pred_scaled = best_model.predict(X_train)
y_test_pred_scaled = best_model.predict(X_test)
print("Train r2:", r2_score(y_train_scaled, y_train_pred_scaled))
print("Test r2:", r2_score(y_test_scaled, y_test_pred_scaled))

Train r2: 0.5151066486918875
Test r2: 0.47444183250401095


# LinearSVR

In [27]:
from sklearn.svm import LinearSVR

In [30]:
best_model = LinearSVR(C=10, epsilon=0.1, max_iter=5000)
best_model.fit(X_train, y_train_scaled)

y_train_pred_scaled = best_model.predict(X_train)
y_test_pred_scaled = best_model.predict(X_test)
print("Train r2:", r2_score(y_train_scaled, y_train_pred_scaled))
print("Test r2:", r2_score(y_test_scaled, y_test_pred_scaled))

Train r2: 0.5151526160361715
Test r2: 0.4739913919729525
